In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, TargetEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

In [2]:
# Load the OKCupid data
# Downloaded on January 18 from https://github.com/rudeboybert/JSE_OkCupid/blob/master/profiles_revised.csv.zip

df = pd.read_csv("profiles_revised.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 59946 entries, 0 to 59945
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   age          59946 non-null  int64  
 1   body_type    54650 non-null  str    
 2   diet         35551 non-null  str    
 3   drinks       56961 non-null  str    
 4   drugs        45866 non-null  str    
 5   education    53318 non-null  str    
 6   ethnicity    54266 non-null  str    
 7   height       59943 non-null  float64
 8   income       59946 non-null  int64  
 9   job          51748 non-null  str    
 10  offspring    24385 non-null  str    
 11  orientation  59946 non-null  str    
 12  pets         40025 non-null  str    
 13  religion     39720 non-null  str    
 14  sex          59946 non-null  str    
 15  sign         48890 non-null  str    
 16  smokes       54434 non-null  str    
 17  speaks       59896 non-null  str    
 18  status       59946 non-null  str    
dtypes: float64(1), 

In [4]:
df.describe()

,age,height,income
count,59946.000000,59943.000000,59946.000000
mean,32.335402,68.295281,20033.222534
std,9.490009,3.994803,97346.192104
min,17.000000,1.000000,-1.000000
25%,26.000000,66.000000,-1.000000
50%,30.000000,68.000000,-1.000000
75%,37.000000,71.000000,-1.000000
max,111.000000,95.000000,1000000.000000


In [5]:
# before we go too far, split!
train, test = train_test_split(df, test_size=0.2, random_state=42)

In [6]:
# How many values in each category?
# AKA cardinality
print("Feature     Count")
# Assuming that str is categorical, could be bad assumption
for feat in train.select_dtypes(include="str").columns:
    print(f"{feat:12}{train[feat].nunique()}")

Feature     Count
body_type   12
diet        18
drinks      6
drugs       3
education   32
ethnicity   202
job         21
offspring   15
orientation 3
pets        15
religion    45
sex         2
sign        48
smokes      5
speaks      6397
status      5


In [8]:
train["speaks"].head(n=20)

8414                                    english (fluently)
35088                                   english (fluently)
24943                                              english
33238                                   english (fluently)
195                                     english (fluently)
47822                                      english, french
42973    english (fluently), french (okay), spanish (po...
2705           english, italian (poorly), spanish (poorly)
22013                   english (okay), italian (fluently)
17466                                  english, vietnamese
23688                                              english
13924                                   english (fluently)
18094                english (fluently), indonesian (okay)
23415                                              english
44445                                english, c++ (poorly)
28061    english (fluently), chinese (fluently), spanis...
47972                          english, french, vietname

In [9]:
train["drinks"].head(n=20)

8414     not at all
35088         often
24943        rarely
33238    not at all
195        socially
47822      socially
42973         often
2705       socially
22013      socially
17466      socially
23688      socially
13924      socially
18094      socially
23415      socially
44445        rarely
28061      socially
47972        rarely
21284      socially
19573      socially
29913      socially
Name: drinks, dtype: str

In [15]:
# look at drinks category
print(train["drinks"].value_counts())

# pass a specific list in the order you want encoded
drink_scale=["not at all", "rarely", "socially", "often", "very often", "desperately"]
drink_enc = OrdinalEncoder(categories=[drink_scale], handle_unknown="use_encoded_value", unknown_value=-1)
drink_ord = drink_enc.fit_transform(train[["drinks"]])
for i in range(10):
    print(train["drinks"].iloc[i], drink_ord[i])

drinks
socially       33512
rarely          4745
often           4100
not at all      2605
very often       386
desperately      258
Name: count, dtype: int64
not at all [0.]
often [3.]
rarely [1.]
not at all [0.]
socially [2.]
socially [2.]
often [3.]
socially [2.]
socially [2.]
socially [2.]


In [ ]:
df["status"].value_counts()
# parameters just to make the display human readable
stat_enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore").set_output(transform="pandas")
stat_one_hot = stat_enc.fit_transform(df[["status"]])
stat_one_hot
# stat_one_hot.head()

,status_available,status_married,status_seeing someone,status_single,status_unknown
0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,1.0,0.0
2,1.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...
59941,0.0,0.0,0.0,1.0,0.0
59942,0.0,0.0,0.0,1.0,0.0
59943,0.0,0.0,0.0,1.0,0.0
59944,0.0,0.0,0.0,1.0,0.0


In [22]:
# What's going with speaks?
train["speaks"].head(10)

8414                                    english (fluently)
35088                                   english (fluently)
24943                                              english
33238                                   english (fluently)
195                                     english (fluently)
47822                                      english, french
42973    english (fluently), french (okay), spanish (po...
2705           english, italian (poorly), spanish (poorly)
22013                   english (okay), italian (fluently)
17466                                  english, vietnamese
Name: speaks, dtype: str

## Following section not yet covered
Feature hashing and target encoding will be addressed at a later date.

In [ ]:
# Drop nans for demonstration purposes
# Dropping nans from one column, needs to apply to the whole dataset
nona_speaks = train["speaks"].dropna()
encoder = FeatureHasher(n_features=8, input_type="string")
encoded_speaks = encoder.fit_transform(nona_speaks.str.split(", "))

In [ ]:
test = ["french (okay)", "spanish (poorly)", "python", "c++", "japanese", "a", "b", "c", "d", "1234","english"]
encoder.transform([test]).todense()

In [ ]:
for s, e in zip(nona_speaks[:20], encoded_speaks[:20]):
    print(f"{s}: {e.todense()}")

In [ ]:
# income vs education
train.groupby("education")["income"].describe()
edu_encoder = TargetEncoder(target_type="continuous")
edu_feat = edu_encoder.fit_transform(train[["education"]],train["income"])

# Look at the target-encoded values
print(f"{'Education':35} Income")
for cat, enc in sorted(zip(edu_encoder.categories_[0], edu_encoder.encodings_[0]), key=lambda e: e[1], reverse=True):
    print(f"{cat:35} ${enc:,.2f}")

In [ ]:
income_buckets = pd.cut(train["income"], bins=[-1000000, 0, 30000, 80000, 1000000], labels=["invalid", "low", "medium", "high"])

for inc, b in zip(train["income"], income_buckets):
    print(f"${inc:,.2f}, {b}")

In [ ]:
income_buckets.value_counts()

## Extra plots
Used in lecture slides, but how they're made isn't very exciting

In [ ]:
# Discretize age
train["u30"] = train["age"] < 30
bins = np.linspace(10,80,40)
ax = train.query("u30 == 1")["age"].hist(label="Under 30", bins=bins)
train.query("u30 == 0")["age"].hist(label="30+", ax=ax, bins=bins)
plt.legend()
plt.xlabel("Age")
plt.ylabel("Frequency")

plt.savefig("../../img/04-discrete.png")